In [ ]:
## imports module
import anndata as ad
import scanpy as sc
import gc, os
import sys
from rich import print
import numpy as np
import psutil
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sea
import harmonypy as hm
import os
import psutil, gc
import matplotlib as mpl

from rich import print

In [ ]:
## general setting
sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80)
pd.set_option('display.max_columns', None)

# check how many memory used
print(psutil.virtual_memory())
# Force garbage collection
gc.collect()  
sc.settings.set_figure_params(dpi=200, facecolor="white")
PATH = "/home/kibr/Work/Vascular_atlas"

## set path and parameters
os.chdir(PATH)
sc.set_figure_params(figsize=(6, 6), frameon=False)

In [ ]:
## --- Load data --- ##
adata = sc.read_h5ad(PATH + "/Results/Revision_2/clean_object_revision_04242026.h5ad")

## subset the adata to only include cell types of interest
adata = adata[adata.obs['Cell_type'].isin(['Pericyte'])].copy()

print(adata)
print(adata.X[:10,:10])

In [ ]:
## save backup
adata.raw = adata.copy()

## filter the features again
sc.pp.filter_genes(adata, min_cells=20) 

## Filtering out the region with low nuclei counts
nuclei_counts = adata.obs['brain_region'].value_counts()
regions_to_keep = nuclei_counts[nuclei_counts >= 50].index.tolist()

## showing the regions were filtered out
print("Regions filtered out due to low nuclei counts:")
print(set(adata.obs['brain_region'].unique()) - set(regions_to_keep))

## subsetting the adata
adata = adata[adata.obs['brain_region'].isin(regions_to_keep)].copy()
print(adata)
print(adata.X[:10,:10])

### Performing another round of integration using Harmony

In [ ]:
## normalization
sc.pp.normalize_total(adata,target_sum=1e6)
sc.pp.log1p(adata)
adata.layers["log1p"] = adata.X.copy()

print(adata.X[:8,:8])

In [ ]:
## to reach the best results, using the below setting
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=1000,
    subset=False,
    layer="log1p",
    flavor="seurat",
    batch_key="individualID",
)

## only using the highly variable genes for integration task
adata = adata[:,adata.var["highly_variable"]].copy()
print(adata)

In [ ]:
### pca and integration
print("Running PCA")
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=50, svd_solver="arpack")
print(psutil.virtual_memory())

In [ ]:
## Harmony integration 
print(adata)
print("Applying Harmony integration")
sc.external.pp.harmony_integrate(adata, key='individualID', max_iter_harmony=50,basis='X_pca')
rep = "X_pca_harmony"

print(psutil.virtual_memory())

In [ ]:
print("Computing neighbors")
rep = "X_pca_harmony"
sc.pp.neighbors(adata, use_rep=rep,metric="cosine")

print("Computing leiden")
sc.tl.leiden(adata,resolution=0.8,key_added='leiden_harmony')

In [ ]:
# Check integration figures for remaining batch(individual effect)
sc.settings.figdir = f"{PATH}/Results/Revision_2/Figures"
os.makedirs(sc.settings.figdir, exist_ok=True)

sc.tl.umap(
    adata,
    # reuse existing neighbors graph
    random_state=42,
    key_added="umap_harmony"
)

sc.pl.embedding(
    adata,
    basis="umap_harmony", 
    color=["leiden_harmony", "individualID",'brain_region','Cell_type'],
    # color=["leiden_harmony", "individualID",'brain_region','Cell_type'],
    frameon=False,
    use_raw=False,
    ncols=4,
    size=20,
    legend_loc="on data",
    # save=f"_Pericyte_umap_harmony.svg"
)

In [ ]:
# # ### Given some known cell type markers, annotate the clusters
# adata = adata.raw.to_adata()
# adata.raw = adata.copy()
# print(adata.X[:10,:10])

# sc.pp.normalize_total(adata,target_sum=1e6)
# sc.pp.log1p(adata)
# sc.pp.scale(adata, max_value=10)

# marker_genes_dict = {
#     "Pericyte": ["RGS5", "PDGFRB",'IL1RAPL1','GRM8'],
#     "SMC": ["ACTA2", "MYH11"],
#     "OLIGO": ["MBP", "MOG", "PLP1"],
#     "Neuron": ["RBFOX3", "SYT1", "GRIN1"],
#     "Endothelial": ["CLDN5", "VWF", "FLT1"],
#     "Astrocyte": ["AQP4", "GFAP", "ALDH1L1"],
# }

# sc.pl.dotplot(adata, marker_genes_dict, groupby='leiden_harmony')

# sc.pl.embedding(
#     adata,
#     basis="umap_harmony", 
#     color=["leiden_harmony", "individualID",'brain_region',"MYH11","SLC6A13",'RGS5'],
#     frameon=False,
#     use_raw=False,
#     ncols=3,
#     size=20,
#     legend_loc="on data",
#     # save=f"_SMC_umap_harmony.svg"
# )

In [ ]:
# sc.settings.figdir = f"{PATH}/Results/Revision_2/Figures"
# # cell types
# cluster2celltype = {
#     "0": "SMC",
#     "1": "SMC",
#     "2": "Pericyte",
#     "3": "SMC",
#     "4": "SMC",
#     "5": "SMC",
#     "6": "SMC",
#     "7": "SMC",
#     "8": "SMC",
# }

# adata.obs["Cell_type"] = adata.obs["leiden_harmony"].map(cluster2celltype)

# sc.pl.embedding(
#     adata,
#     basis='umap_harmony',   # <- THIS picks .obsm['X_' + key]
#     color=['leiden_harmony','individualID','Cell_type',"MYH11","SLC6A13",'SLC6A1'],
#     frameon=False,
#     ncols=3,
#     size=20,
#     legend_loc="on data",
#     use_raw = False,
#     save=f"_Pericyte_umap_harmony.svg"
#     )


In [ ]:
# marker_genes_dict = {
#     "Pericyte": ["RGS5", "PDGFRB",'IL1RAPL1','GRM8','SLC6A12','SLC6A1',"HIGD1B"],
#     "SMC": ["ACTA2", "MYH11"],
# }

# # sc.pl.dotplot(adata, marker_genes_dict, groupby='leiden_harmony')

# sc.pl.dotplot(adata, marker_genes_dict, groupby='Cell_type',
#               categories_order=['Pericyte', 'SMC'],
#               use_raw=False)

## Builting region-level representation (Donor-balanced centroids in PC spave)

In [ ]:
# Choose how many PCs to represent each cell
rep = "X_pca_harmony"
n_pcs_use = 30
Xpc = adata.obsm[rep][:, :n_pcs_use]

# centroid for each (region, donor) group
df_index = adata.obs[["brain_region", "individualID"]].copy()
df_index["cell_idx"] = np.arange(adata.n_obs)

# Compute group centroids efficiently
# We'll build a table of centroids: rows = (region, donor), cols = PC1..PCn
pcs = pd.DataFrame(Xpc, index=adata.obs_names, columns=[f"PC{i+1}" for i in range(n_pcs_use)])
pcs["brain_region"] = adata.obs["brain_region"].values
pcs["individualID"]  = adata.obs["individualID"].values

rg_dn_centroids = pcs.groupby(["brain_region", "individualID"]).mean()
# print(rg_dn_centroids.head())

## donor-balanced region centroid
rg_centroids = rg_dn_centroids.groupby(level=0).mean()

# Optional: track how many donors contribute per region (useful QC)
donors_per_region = rg_dn_centroids.reset_index().groupby("brain_region")["individualID"].nunique()

## Create region -level AnnData and run leiden on region graph
adata_rg = sc.AnnData(X=rg_centroids.values)
adata_rg.obs_names = rg_centroids.index.astype(str)
adata_rg.var_names = [f"PC{i+1}" for i in range(n_pcs_use)]

adata_rg.obs["n_donors"] = donors_per_region.reindex(adata_rg.obs_names).fillna(0).astype(int).values

print(adata_rg)

## Adding meta data

In [ ]:
## Get the region ID and region name from the obs_names
adata_rg.obs["region_ID"] = (
    adata_rg.obs_names
        .astype(str)
        .str.split("_", n=1)
        .str[0]
)
adata_rg.obs["region_name"] = (
    adata_rg.obs_names
        .astype(str)
        .str.split("_", n=1)
        .str[1]
)

## read the region metadata
df = pd.read_csv(PATH + '/Data/region_update.csv')
df.head()



In [ ]:
# check for mismatches
df["regionID"] = df["regionID"].astype(str).str.strip()
adata_rg.obs["region_ID"] = adata_rg.obs["region_ID"].astype(str).str.strip()

missing = set(adata_rg.obs["region_ID"]) - set(df["regionID"])
print("Region IDs missing from metadata:", missing)

## adding annotation
adata_rg.obs["Region_layer"] = (
    adata_rg.obs["region_ID"]
        .map(df.set_index("regionID")["Region_layer_2"])
)

adata_rg.obs["Region_full_name"] = (
    adata_rg.obs["region_ID"]
        .map(df.set_index("regionID")["Region_layer_1"])
)

adata_rg.obs["region_abb"] = (
    adata_rg.obs["region_ID"]
        .map(df.set_index("regionID")["Final_abb"])
)

## Capitalize the region names for better visualization
adata_rg.obs["Region_full_name"] = adata_rg.obs["Region_full_name"].str.title()
## string combine region_ID and region_name for better visualization
adata_rg.obs["Region_combined"] = adata_rg.obs["region_ID"] + "_" + adata_rg.obs["Region_full_name"]
adata_rg.obs["Region_combined_2"] = adata_rg.obs["region_ID"] + "_" + adata_rg.obs["region_name"]

adata_rg.obs.set_index('Region_combined', inplace=True)

adata_rg.obs["Region_combined"] = adata_rg.obs.index.copy()

adata_rg.obs.head()

### Clustering and plotting

In [ ]:
# Build kNN graph across ~40 regions
# With only 40 nodes, keep neighbors modest (e.g., 5–15)
sc.pp.neighbors(adata_rg, n_neighbors=5, n_pcs=n_pcs_use, use_rep="X")

# Leiden clustering of regions (sweep resolution later)
sc.tl.leiden(adata_rg, resolution=1.5, key_added="leiden")

# 2D embedding for visualization 
sc.tl.umap(adata_rg, min_dist=0.4)

sc.tl.dendrogram(adata_rg, groupby="leiden")

## working on the dot size based on nuclei counts
### Clean the region names for better visualization
adata.obs["regionID"] = adata.obs["regionID"].astype(str).str.strip()
adata.obs["region_name"] = (adata.obs["regionID"].map(df.set_index("regionID")["Region_layer_1"]))

## Capitalize the region names for better visualization
adata.obs["region_name"] = adata.obs["region_name"].str.title()
## string combine regionID and region_name for better visualization
adata.obs["Region_combined"] = adata.obs["regionID"].astype(str) + "_" + adata.obs["region_name"]
## Get the nuclei counts for each region
nuclei_counts = adata.obs['Region_combined'].value_counts()

## ordering counts based on adata_rg.obs['Region_combined']
nuclei_counts_dict = nuclei_counts.to_dict()

# Size parameters might require scaling; you can adjust the scaling factor as needed
size_factor = 2  # Adjust this factor to change the size scale
dot_sizes = np.array([nuclei_counts_dict[region] * size_factor for region in adata_rg.obs.index])
print(nuclei_counts.head())

In [ ]:
# ----------------------------
# Plotting (region-level)
# ----------------------------
mpl.rcParams['svg.fonttype'] = 'none'
sc.set_figure_params(figsize=(8, 8), frameon=False)

sc.pl.umap(
    adata_rg,
    color="leiden",
    size=dot_sizes,
    legend_loc="on data",
    show=False
)

# add region labels
ax = plt.gca()
X = adata_rg.obsm["X_umap"]

for i, region in enumerate(adata_rg.obs_names):
    label = region.split("_", 1)[-1]
    ax.text(
        X[i, 0],
        X[i, 1],
        label,
        fontsize=8,
        ha="center",
        va="center"
    )

plt.show()

## UMAP of region_layer
sc.pl.umap(
    adata_rg,
    color="Region_layer",
    size=dot_sizes,
    legend_loc="right margin",
    show=False
)

# add region labels
ax = plt.gca()
X = adata_rg.obsm["X_umap"]

for i, region in enumerate(adata_rg.obs_names):
    label = region.split("_", 1)[-1]
    ax.text(
        X[i, 0],
        X[i, 1],
        label,
        fontsize=8,
        ha="center",
        va="center"
    )

plt.show()

# sc.pl.umap(adata_rg, color=["region_name"],legend_loc = "on data")
sc.pl.dendrogram(adata_rg, groupby="leiden",)

In [ ]:
## Option 2
# ----------------------------
# 1) Define clusters (leiden -> region_cluster)
# ----------------------------
cluster_map = {
    "0": "Cluster_3",
    "1": "Cluster_1",
    "2": "Cluster_3",
    "3": "Cluster_1",
    "4": "Cluster_2",
    "5": "Cluster_3",
    "6": "Cluster_2",
}

adata_rg.obs["region_cluster"] = adata_rg.obs["leiden"].astype(str).map(cluster_map).fillna("Unassigned").astype("category")

# ----------------------------
# 2) Define Region_layer colors (your palette)
# ----------------------------
region_color_map = {
    "Cortex": "#009E73",
    "Brainstem": "#D55E00",
    "Cerebellum": "#046299",
    "Limbic": "#03B8D8",
    "Watershed": "#E69F00",
    "White Matter Tracts": "#CC79A7",
    "Olfactory": "#999999",
    "Barrier": "#9C0AE0",
    # "Subcortical": "#915330",
    "Major Vessel": "#FF0000",
}


# Ensure categorical for stable color ordering
adata_rg.obs["Region_layer"] = adata_rg.obs["Region_layer"].astype("category")

# Scanpy expects a list of colors aligned to category order
adata_rg.uns["Region_layer_colors"] = [
    region_color_map.get(cat, "#BDBDBD")  
    for cat in adata_rg.obs["Region_layer"].cat.categories
]

# ----------------------------
# 3) Plot UMAP colored by Region_layer
# ----------------------------
sc.set_figure_params(figsize=(9, 9), frameon=False)

sc.pl.umap(
    adata_rg,
    color="Region_layer",
    size=dot_sizes,
    legend_loc="right margin",
    show=False
)

ax = plt.gca()
X = adata_rg.obsm["X_umap"]

# ----------------------------
# 4) Add region labels (after the first "_")
# ----------------------------
for (x, y), region in zip(X, adata_rg.obs_names):
    label = adata_rg.obs[adata_rg.obs_names == region]['region_abb'].astype(str).values[0]
    ax.text(
        x, y, label,
        fontsize=10,
        ha="center",
        va="center",
        clip_on=True
    )
# ----------------------------
# 5) Add KDE contours per region_cluster + label each contour
# ----------------------------
clusters = adata_rg.obs["region_cluster"].astype(str)

# Tuning knobs for "closer" contours:
bw_adjust = 1.2   # smaller -> tighter / more detailed
thresh = 0.15     # larger -> tighter (keeps higher-density region)
alpha = 0.1  # smaller -> more transparent contours (can help with overlap)

for cl in sorted(clusters.unique()):
    if cl == "Unassigned":
        continue

    idx = (clusters == cl).values
    pts = X[idx, :]

    # KDE unstable for very small clusters
    if pts.shape[0] < 3:
        continue

    sea.kdeplot(
        x=pts[:, 0],
        y=pts[:, 1],
        fill=True,          # << shaded region
        levels=4,           # single filled region
        bw_adjust=bw_adjust,
        thresh=thresh,
        alpha=alpha,
        linewidth=0,
        ax=ax
    )

    # Optional: cluster label at center
    cx, cy = np.median(pts[:, 0]), np.median(pts[:, 1])
    ax.text(
        cx, cy, cl,
        fontsize=11,
        ha="center",
        va="center",
        bbox=dict(boxstyle="round,pad=0.2", alpha=0.6, linewidth=0),
        clip_on=True
    )

# # Optional cleanup
# ax.set_xticks([])
# ax.set_yticks([])
# ax.set_aspect("equal")

plt.savefig("./Results/Revision_2/Figures/umap_region_clusters_abb_Pericyte.svg",format="svg",bbox_inches="tight",dpi=600)
plt.show()

In [ ]:
### ---------------------------------------------------------------------
### Saving processed data
### ---------------------------------------------------------------------
adata = adata.raw.to_adata()
print(adata)
print(adata.X[:10,:10])
## Saving h5ad
results_file = PATH+"/Results/Revision_2/Pericyte_temp_processed.h5ad"
adata.write(results_file)
print("Done. Saved:", results_file)